In [1]:

# Run this in a notebook cell to reinstall matching torch/torchvision stacks 
%uv pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
%uv pip install transformers peft trl bitsandbytes accelerate datasets
%uv pip install -q peft trl bitsandbytes datasets scikit-learn
%uv pip install --no-build-isolation flash-linear-attention causal_conv1d

Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 18ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 75 packages in 338ms
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
click      ------------------------------     0 B/113.91 KiB
⠙ Preparing packages... (0/16)
click      ------------------------------ 14.84 KiB/113.91 KiB
⠙ Preparing packages... (0/16)
typer      ------------------------------     0 B/57.04 KiB
click      ------------------------------ 14.84 KiB/113.91 KiB
⠙ Preparing packages... (0/16)
typer      ------------------------------ 14.84 KiB/57.04 KiB
click      ------------------------------ 14.84 KiB/113.91 KiB
⠙ Preparing packages... (0/16)
typer      ------------------------------ 14.84 KiB/57.04 KiB
click      ------------------------------ 14.84 KiB/113.91 KiB
⠙ Preparing packages... (0/16)
typer      ---------------------------

In [2]:
import os
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])  # never hardcode tokens in notebooks


In [3]:
import torch
import torchvision
print("Torch CUDA available:", torch.cuda.is_available())

# This should now import without issues
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
print("Imports successful!")

Torch CUDA available: True
Imports successful!


In [4]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoProcessor
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from tqdm.notebook import tqdm

# 1. CONFIGURATION & MODEL INITIALIZATION

In [5]:
# Switched from Gemma-3 to Liquid Foundation Model (LFM2.5)
model_id = "LiquidAI/LFM2.5-350M"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading {model_id} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16 # Using standard torch_dtype
)

# Load the LFM tokenizer directly
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Synchronize special token padding allocations across the configs
# NOTE: LFM templates natively handle padding through eos_token / im_end
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.eos_token_id

Loading LiquidAI/LFM2.5-350M in 4-bit...


config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

In [6]:
# --- 2. DATASET INGESTION & CHATML REFORMATTING ---
dataset = load_dataset("ryuma007/hishab-synthetic-v2")

train_dataset = dataset["train"]
val_dataset   = dataset["validation"]

SYSTEM_PROMPT = """
You are a precise Bangladeshi financial transaction parser. Your task is to process inputs in English, Bengali, or Banglish and extract financial records in a structured two-phase sequence:

Phase 1: Output "TRANSLATION: " followed by a clean English translation of the user's text. Ensure all implicit currency values are standardized as BDT, and explicitly expand financial shorthand units (e.g., convert "4.5k" or "15k" directly into raw integers like "4500" or "15000").
Phase 2: Output "JSON:" followed immediately by a valid, minified JSON array containing the transaction objects. 

Each transaction object in the JSON list must strictly adhere to these key structures:
- "description" (string, in English or native script mapping the item/source)
- "amount" (float/integer value of the transaction)
- "category" (string classification: e.g., food, grocery, entertainment, shopping, utility, salary, transport, tuition, gift)
- "type" (string: either "expense" or "income")

Strict Rule: Do not add conversational filler, markdown code-blocks, or introductory commentary. Adhere strictly to the two-phase sequential format.
"""

def convert_to_lfm(example):
    # LFM uses the industry standard multi-turn list dictionary array layout 
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]}
        ]
    }

train_dataset = train_dataset.map(convert_to_lfm, remove_columns=["instruction", "response", "id", "language"])
val_dataset   = val_dataset.map(convert_to_lfm,   remove_columns=["instruction", "response", "id", "language"])

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print("Sample formatted for LFM Chat Template:")
print(train_dataset[0]["messages"])

README.md:   0%|          | 0.00/499 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/707k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/32.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5200 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/5200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Train: 5200 | Val: 200
Sample formatted for LFM Chat Template:
[{'role': 'system', 'content': '\nYou are a precise Bangladeshi financial transaction parser. Your task is to process inputs in English, Bengali, or Banglish and extract financial records in a structured two-phase sequence:\n\nPhase 1: Output "TRANSLATION: " followed by a clean English translation of the user\'s text. Ensure all implicit currency values are standardized as BDT, and explicitly expand financial shorthand units (e.g., convert "4.5k" or "15k" directly into raw integers like "4500" or "15000").\nPhase 2: Output "JSON:" followed immediately by a valid, minified JSON array containing the transaction objects. \n\nEach transaction object in the JSON list must strictly adhere to these key structures:\n- "description" (string, in English or native script mapping the item/source)\n- "amount" (float/integer value of the transaction)\n- "category" (string classification: e.g., food, grocery, entertainment, shopping, util

In [7]:
import json
import re
import torch
from tqdm import tqdm

def parse_model_output(output_str: str):
    """
    Splits the raw string into translation and safely parsed JSON objects.
    Handles variations in whitespaces or case markers.
    """
    translation = ""
    json_data = None
    
    # Extract translation block
    if "TRANSLATION:" in output_str:
        # Split by JSON separator to get the text between translation and JSON
        parts = re.split(r"JSON:\s*", output_str, maxsplit=1)
        translation = parts[0].replace("TRANSLATION:", "").strip()
        
        # Extract and parse the JSON block if present
        if len(parts) > 1 and parts[1].strip():
            json_str = parts[1].strip()
            try:
                json_data = json.loads(json_str)
            except json.JSONDecodeError:
                json_data = None # Invalid JSON generated
    return translation, json_data

def evaluate_model(model_to_eval, tokenizer, eval_split, num_samples=50, system_prompt=""):
    model_to_eval.eval()
    
    valid_json_count = 0
    perfect_json_matches = 0
    total = min(len(eval_split), num_samples)

    print(f"\nEvaluating on {total} samples...")
    # Select a slice of the dataset
    subset = eval_split.select(range(total))

    for example in tqdm(subset):
        # 1. Map from our lean dataset keys
        user_input   = example["instruction"]
        ground_truth = example["response"]
        
        # Parse ground truth strings into concrete segments
        gt_translation, gt_json = parse_model_output(ground_truth)

        # 2. Re-create the exact LFM ChatML sequence
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
        
        tokenized_chat = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to("cuda")

        # 3. Generate tokens safely using LFM recommended inference patterns
        with torch.inference_mode():
            outputs = model_to_eval.generate(
                **tokenized_chat,
                max_new_tokens=256,  # Supports translation text + multi-item arrays
                do_sample=False,     # Deterministic greedy decoding for stable metrics
                pad_token_id=tokenizer.eos_token_id
            )

        input_length     = tokenized_chat["input_ids"].shape[1]
        generated_tokens = outputs[0][input_length:]
        prediction       = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        # Parse generated string elements
        pred_translation, pred_json = parse_model_output(prediction)

        # 4. Transaction Object Structural Evaluation
        if pred_json is not None:
            valid_json_count += 1
            
            # Compare the python data objects directly (order/whitespace insensitive)
            if pred_json == gt_json:
                perfect_json_matches += 1

    print("\n" + "="*40)
    print(f"Evaluation Complete across {total} samples:")
    print(f"Valid JSON Syntactical Rate: {valid_json_count / total * 100:.2f}%")
    print(f"Semantic JSON Match Accuracy: {perfect_json_matches / total * 100:.2f}%")
    print("="*40)

In [8]:
# --- 4. FINETUNING CONFIGURATION VIA TRL ---
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# FIX 1: Map the target modules to LFM2.5's hybrid architecture
# Standard Llama/Gemma projection names will cause NaN gradient calculations here.
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "out_proj", # Attention layers
        "in_proj",                                # Convolution components
        "w1", "w2", "w3"                          # Feed-Forward Network components
    ]
)

training_args = SFTConfig(
    output_dir="./lfm2_5-350M-hishab-v2",

    # --- A100 SPEED & SCALE OPTIMIZATIONS ---
    per_device_train_batch_size=64,
    gradient_accumulation_steps=1,
    dataloader_num_workers=4,

    learning_rate=2e-4,
    num_train_epochs=5,           
    warmup_steps=0.05,
    lr_scheduler_type="cosine",

    logging_steps=7,
    eval_strategy="epoch",        
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    bf16=True,
    packing=False,
    max_length=512,
    report_to="none"
)

# FIX 2: Remove 'dataset_text_field' from SFTConfig. 
# SFTTrainer automatically handles formatting when it sees a column named "messages".
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,     
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer
)

trainer.train()

# Save best LoRA adapters
trainer.model.save_pretrained("lfm2_5-350M-hishab-v2-adapters")
tokenizer.save_pretrained("lfm2_5-350M-hishab-v2-adapters")

Tokenizing train dataset:   0%|          | 0/5200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.206897,0.197460,0.203770,2157026.000000,0.950136
2,0.176343,0.156535,0.171380,4314052.000000,0.958917
3,0.154182,0.136782,0.151438,6471078.000000,0.963690
4,0.136526,0.128522,0.140562,8628104.000000,0.965675
5,0.146206,0.127864,0.138281,10785130.000000,0.965965


('lfm2_5-350M-hishab-v2-adapters/tokenizer_config.json',
 'lfm2_5-350M-hishab-v2-adapters/chat_template.jinja',
 'lfm2_5-350M-hishab-v2-adapters/tokenizer.json')

In [9]:
# --- 5. POST-TRAINING EVALUATION ---
import torch
import json
import re
from tqdm import tqdm

def parse_model_output(output_str: str):
    """Splits the raw string into translation and safely parsed JSON objects."""
    translation = ""
    json_data = None
    if "TRANSLATION:" in output_str:
        parts = re.split(r"JSON:\s*", output_str, maxsplit=1)
        translation = parts[0].replace("TRANSLATION:", "").strip()
        if len(parts) > 1 and parts[1].strip():
            try:
                json_data = json.loads(parts[1].strip())
            except json.JSONDecodeError:
                json_data = None
    return translation, json_data

def evaluate_model_on_messages(model_to_eval, tokenizer, eval_split, num_samples=100):
    """Evaluates the model by feeding it the exact system and user blocks from Cell 2."""
    model_to_eval.eval()
    valid_json_count = 0
    perfect_json_matches = 0
    total = min(len(eval_split), num_samples)

    print(f"Evaluating model on {total} samples directly from the validation split...")
    subset = eval_split.select(range(total))

    for example in tqdm(subset):
        chat_history = example["messages"]
        
        # Isolate system + user messages to send to the model input
        input_messages = chat_history[:-1]
        
        # Isolate ground truth assistant string (Translation + JSON)
        ground_truth = chat_history[-1]["content"].strip()
        gt_translation, gt_json = parse_model_output(ground_truth)

        # Tokenize with exact template tokens applied during training
        tokenized_chat = tokenizer.apply_chat_template(
            input_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to("cuda")

        with torch.inference_mode():
            outputs = model_to_eval.generate(
                **tokenized_chat,
                max_new_tokens=256, # Generous limit for complex lists
                do_sample=False,    # Greedy decoding for stable evaluation metrics
                pad_token_id=tokenizer.eos_token_id
            )

        input_length     = tokenized_chat["input_ids"].shape[1]
        generated_tokens = outputs[0][input_length:]
        prediction       = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        # Parse generated string metrics
        pred_translation, pred_json = parse_model_output(prediction)

        if pred_json is not None:
            valid_json_count += 1
            if pred_json == gt_json:
                perfect_json_matches += 1

    print("\n" + "="*40)
    print(f"Post-Fine-Tuning Evaluation Complete:")
    print(f"Valid JSON Syntax Rate:      {valid_json_count / total * 100:.2f}%")
    print(f"Semantic JSON Match Accuracy: {perfect_json_matches / total * 100:.2f}%")
    print("="*40)

# Run evaluation on the active training model state
trained_model = trainer.model
print("\n--- Post-fine-tuning evaluation ---")
evaluate_model_on_messages(trained_model, tokenizer, val_dataset, num_samples=100)


--- Post-fine-tuning evaluation ---
Evaluating model on 100 samples directly from the validation split...


100%|████████████████████████████████████████████████████████████████████████| 100/100 [06:10<00:00,  3.71s/it]


Post-Fine-Tuning Evaluation Complete:
Valid JSON Syntax Rate:      100.00%
Semantic JSON Match Accuracy: 35.00%


In [10]:
# --- 6. VISUAL INFERENCE LOOP ---
import torch
from datasets import load_dataset

print("Fetching raw validation split for side-by-side comparison...")
# Pull from the correct validation set for the Hishab project
raw_val = load_dataset("ryuma007/hishab-synthetic-v2", split="validation")

trained_model.eval()

# Check the first 8 samples to see the language mix behavior
for idx in range(8):
    example      = raw_val[idx]
    instruction  = example["instruction"]
    ground_truth = example["response"]
    lang         = example.get("language", "unknown")

    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": instruction}
    ]

    inputs = tokenizer.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    with torch.inference_mode():
        outputs = trained_model.generate(
            **inputs,
            max_new_tokens=256, # Raised to 256 so multi-item JSON arrays don't cut off
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    prompt_length    = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][prompt_length:]
    prediction       = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    print("\n" + "="*70)
    print(f"SAMPLE #{idx + 1} | LANGUAGE TYPE: {lang.upper()}")
    print("="*70)
    print(f"👉 USER REQUEST:\n   {instruction}")
    print(f"\n✅ GROUND TRUTH:\n   {ground_truth}")
    print(f"\n🤖 MODEL OUTPUT:\n   {prediction}")

    # Check if the text matches exactly
    if prediction.strip() == ground_truth.strip():
        print(f"\nSTATUS: 🎉 PERFECT STRING MATCH")
    else:
        # Check if it at least contains a valid JSON block structural breakdown
        if "JSON:" in prediction:
            print(f"\nSTATUS: ⚠️ STRING MISMATCH (Review translation text or JSON whitespace differences)")
        else:
            print(f"\nSTATUS: ❌ STRUCTURAL BREAKDOWN (Model missed JSON marker or failed structure)")

print("\n" + "="*70)

Fetching raw validation split for side-by-side comparison...

SAMPLE #1 | LANGUAGE TYPE: BANGLISH
👉 USER REQUEST:
   Bkash send money to my sister 2000 taka.

✅ GROUND TRUTH:
   TRANSLATION: Send 2000 BDT to my sister via bKash.
JSON:
[{"description":"Bkash send money to sister","amount":2000.0,"category":"gift","type":"expense"}]

🤖 MODEL OUTPUT:
   TRANSLATION: Send 2000 BDT to my sister via bKash.
JSON:
[{"description":"Bkash send money to sister","amount":2000.0,"category":"gift","type":"expense"}]

STATUS: 🎉 PERFECT STRING MATCH

SAMPLE #2 | LANGUAGE TYPE: BANGLISH
👉 USER REQUEST:
   Ajke doctor dekhalam 1000 tk, tarpor 300 tk dawa kinlam.

✅ GROUND TRUTH:
   TRANSLATION: I saw a doctor today for 1000 taka and then bought medicine for 300 taka.
JSON:
[{"description":"doctor visit","amount":1000.0,"category":"medical","type":"expense"},{"description":"medicine","amount":300.0,"category":"medical","type":"expense"}]

🤖 MODEL OUTPUT:
   TRANSLATION: I saw a doctor today for 1000 taka

In [11]:
# --- 7. MANUAL MULTI-LINGUAL STRESS TESTS ---
manual_prompts = [
    # 1. Pure Banglish with numeric shorthand and mixed text
    "Wall clock kinlam 1200, ar ekta photo frame 600, ar tuition theke pelam 4.5k",
    
    # 2. Native Bengali with native digits and explicit currency text
    "আজকের বাজার: মাছ ৫০০, মাংস ৮০০, সবজি ৩০০, ফল ১৫০। সব মিলিয়ে মোট কত হলো?",
    
    # 3. Heavy phonetics / typical casual text
    "Ami 300 taka diye bhat kheyechi hotel e.",
    
    # 4. Complex multi-item mix with income and expense strings
    "Eidl er special khabar 1000, salon service 600, r amake ekta gift dilo 2000",
    
    # 5. Pure English baseline string
    "Spent 3000 on sightseeing tours and another 1000 for souvenirs.",
    
    # 6. Casual conversation/slang variant
    "Bhai, 500 taka flexiload korsi r dosto 1200 taka dhar ferot dise",
    
    # 7. Heavy code-switching (English words in Roman character style)
    "Office er desk repair koraite cost holo 3500 BDT, lunch e spent 450",

    "Ajke vat khelam 30 takar, icecream hoilo 40 taka ebong tuition theke pelam 3.5k",
    
    # 8. High value native conversion test
    "বাবার চিকিৎসার জন্য ১৫ হাজার টাকা খরচ হইলো আর জমানো ডিপিএস থেইকা পাইলাম ৫০কে"
]

print("Executing manual evaluation scenario matrices...")
for i, test_prompt in enumerate(manual_prompts):
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": test_prompt}
    ]
    
    inputs = tokenizer.apply_chat_template(conversation, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    
    with torch.inference_mode():
        outputs = trained_model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        
    prompt_length = inputs["input_ids"].shape[1]
    prediction = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True).strip()
    
    print("\n" + "-"*50)
    print(f"TEST #{i+1}: {test_prompt}")
    print("-"*50)
    print(f"{prediction}")

Executing manual evaluation scenario matrices...

--------------------------------------------------
TEST #1: Wall clock kinlam 1200, ar ekta photo frame 600, ar tuition theke pelam 4.5k
--------------------------------------------------
TRANSLATION: I bought a wall clock for 1200, a photo frame for 600, and received 4500 from tuition.
JSON:
[{"description":"Wall clock","amount":1200.0,"category":"shopping","type":"expense"}, {"description":"Photo frame","amount":600.0,"category":"shopping","type":"expense"}, {"description":"Tuition","amount":4500.0,"category":"tuition","type":"income"}]

--------------------------------------------------
TEST #2: আজকের বাজার: মাছ ৫০০, মাংস ৮০০, সবজি ৩০০, ফল ১৫০। সব মিলিয়ে মোট কত হলো?
--------------------------------------------------
TRANSLATION: Today's grocery list: fish 50 taka, meat 800 taka, vegetables 300 taka, fruit 150 taka. I need to top up my money.
JSON:
[{"description":"মাছ","amount":50.0,"category":"grocery","type":"expense"}, {"descript

In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Switched to your LFM configuration
base_model_id = "LiquidAI/LFM2.5-350M" 
adapter_dir = "lfm2_5-350M-hishab-v2-adapters"
merged_dir = "lfm2_5-350M-hishab-v2-merged"

print("🔄 Loading LFM base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.bfloat16, # Recommended over float16 for LFM to prevent precision truncation
    device_map="auto"
)

print("🔗 Attaching your fine-tuned LFM LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print("🛠️ Merging weights into a single model...")
# PEFT handles merge_and_unload for hybrid models natively
merged_model = model.merge_and_unload()

print(f"💾 Saving the complete merged model to disk at '{merged_dir}'...")
merged_model.save_pretrained(merged_dir)

# --- TOKENIZER SANITIZATION ---
print("🧼 Fetching a clean vanilla LFM tokenizer to avoid local path traces...")
clean_tokenizer = AutoTokenizer.from_pretrained(base_model_id)
clean_tokenizer.save_pretrained(merged_dir)

print("✅ LFM Model merged and tokenizer sanitized locally!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


🔄 Loading LFM base model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

🔗 Attaching your fine-tuned LFM LoRA adapters...
🛠️ Merging weights into a single model...
💾 Saving the complete merged model to disk at 'lfm2_5-350M-hishab-v2-merged'...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🧼 Fetching a clean vanilla LFM tokenizer to avoid local path traces...
✅ LFM Model merged and tokenizer sanitized locally!


In [17]:
from huggingface_hub import HfApi

api = HfApi()

hf_username = "ryuma007" 
repo_id = f"{hf_username}/lfm2.5-350M-hishab-synthetic-v2x"

print(f"🚀 Creating Hugging Face repository: {repo_id}")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

print("📤 Uploading sanitized merged model weights and tokenizer configurations...")
api.upload_folder(
    folder_path=merged_dir,
    repo_id=repo_id,
    repo_type="model"
)
print(f"🎉 Success! Your clean model is now live at: https://huggingface.co/{repo_id}")

🚀 Creating Hugging Face repository: ryuma007/lfm2.5-350M-hishab-synthetic-v2x
📤 Uploading sanitized merged model weights and tokenizer configurations...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🎉 Success! Your clean model is now live at: https://huggingface.co/ryuma007/lfm2.5-350M-hishab-synthetic-v2x


In [15]:
!sed -i 's/"TokenizersBackend"/"LlamaTokenizer"/g' ./lfm2_5-350M-hishab-v2-merged/tokenizer_config.json